In [56]:
#https://techvidvan.com/tutorials/python-sentiment-analysis/
#Process data and calculate sentiment column
import pandas as pd
dataframe = pd.read_csv('Resources/Reviews.csv')
dataframe = dataframe[dataframe['Score'] != 3]
dataframe['Sentiment'] = dataframe['Score'].apply(lambda rating : +1 if rating > 3 else -1)
dataframe.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Sentiment
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,1
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,-1
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,1
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...,-1
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,1


In [57]:
#Grab needed columns
review_dataframe = dataframe[['Text', 'Sentiment']]
print(review_dataframe.shape)
review_dataframe.head(5)

(525814, 2)


,Text,Sentiment
0,I have bought several of the Vitality canned d...,1
1,Product arrived labeled as Jumbo Salted Peanut...,-1
2,This is a confection that has been around a fe...,1
3,If you are looking for the secret ingredient i...,-1
4,Great taffy at a great price. There was a wid...,1


In [58]:
# Change Sentiment column to string based on score
review_dataframe['Sentiment'] = review_dataframe['Sentiment'].replace({-1 : 'negative'})
review_dataframe['Sentiment'] = review_dataframe['Sentiment'].replace({1 : 'positive'})
review_dataframe["Sentiment"].value_counts()

/var/folders/5j/wzp0d2c91b7g440s5lnymfm80000gn/T/ipykernel_5995/538753188.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  review_dataframe['Sentiment'] = review_dataframe['Sentiment'].replace({-1 : 'negative'})
/var/folders/5j/wzp0d2c91b7g440s5lnymfm80000gn/T/ipykernel_5995/538753188.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  review_dataframe['Sentiment'] = review_dataframe['Sentiment'].replace({1 : 'positive'})


positive    443777
negative     82037
Name: Sentiment, dtype: int64

In [59]:
# Convert the categorical values to numeric so model can understand it
sentiment_label = review_dataframe.Sentiment.factorize()
sentiment_label

(array([0, 1, 0, ..., 0, 0, 0]),
 Index(['positive', 'negative'], dtype='object'))

In [60]:
#Get text from dataset
review = review_dataframe.Text.values

In [61]:
#Break down all the words/sentences of a text into small parts & associate token with words
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(review)
vocab_size = len(tokenizer.word_index) + 1

In [62]:
#Replace the words with their assigned numbers
encoded_docs = tokenizer.texts_to_sequences(review)

In [63]:
#Pad the sentences to have equal length
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_sequence = pad_sequences(encoded_docs, maxlen=200)

In [64]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.layers import Embedding
embedding_vector_length = 32
model = Sequential()
model.add(Embedding(vocab_size, embedding_vector_length, input_length=200))
model.add(SpatialDropout1D(0.25))
model.add(LSTM(50, dropout=0.5, recurrent_dropout=0.5))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam', metrics=['accuracy'])
print(model.summary())

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 200, 32)           4074304   
                                                                 
 spatial_dropout1d (SpatialD  (None, 200, 32)          0         
 ropout1D)                                                       
                                                                 
 lstm (LSTM)                 (None, 50)                16600     
                                                                 
 dropout (Dropout)           (None, 50)                0         
                                                                 
 dense (Dense)               (None, 1)                 51        
                                                                 
Total params: 4,090,955
Trainable params: 4,090,955
Non-trainable params: 0
____________________________________________

In [65]:
#Train the sentiment model
history = model.fit(padded_sequence,sentiment_label[0],validation_split=0.2, epochs=5, batch_size=32)

Epoch 1/5
13146/13146 [==============================] - 2433s 185ms/step - loss: 0.2026 - accuracy: 0.9213 - val_loss: 0.1555 - val_accuracy: 0.9380
Epoch 2/5
13146/13146 [==============================] - 2232s 170ms/step - loss: 0.1515 - accuracy: 0.9419 - val_loss: 0.1248 - val_accuracy: 0.9525
Epoch 3/5
13146/13146 [==============================] - 2787s 212ms/step - loss: 0.1331 - accuracy: 0.9498 - val_loss: 0.1225 - val_accuracy: 0.9545
Epoch 4/5
13146/13146 [==============================] - 2755s 210ms/step - loss: 0.1232 - accuracy: 0.9538 - val_loss: 0.1162 - val_accuracy: 0.9575
Epoch 5/5
13146/13146 [==============================] - 2668s 203ms/step - loss: 0.1172 - accuracy: 0.9560 - val_loss: 0.1125 - val_accuracy: 0.9588
